In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

ARTIFACTS = Path("../artifacts")
RESULTS = Path("../results")

df = pd.read_csv(ARTIFACTS / "clean_data.csv")
with open(ARTIFACTS / "split_config.json") as f:
    SEED = json.load(f)["random_state"]
with open(ARTIFACTS / "best_params.json") as f:
    best_params = json.load(f)

TARGET = "CO2 uptake (mmol/g)"
FEATURES = [c for c in df.columns if c not in ["row_id", TARGET, "split"]]

train = df[df["split"] == "train"]
val = df[df["split"] == "validation"]
X_train, y_train = train[FEATURES], train[TARGET]
X_val, y_val = val[FEATURES], val[TARGET]

cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
print(X_train.shape, X_val.shape)

(336, 10) (84, 10)


In [2]:
MODEL_CLASSES = {
    "Random Forest":     (RandomForestRegressor,     {"random_state": SEED, "n_jobs": -1}),
    "Extra Trees":       (ExtraTreesRegressor,       {"random_state": SEED, "n_jobs": -1}),
    "Gradient Boosting": (GradientBoostingRegressor, {"random_state": SEED}),
    "XGBoost":           (XGBRegressor,              {"random_state": SEED, "n_jobs": -1}),
    "LightGBM":          (LGBMRegressor,             {"random_state": SEED, "n_jobs": -1, "verbose": -1}),
    "CatBoost":          (CatBoostRegressor,         {"random_state": SEED, "verbose": 0}),
}

def make_model(name):
    cls, fixed = MODEL_CLASSES[name]
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("model", cls(**best_params[name], **fixed)),
    ])

models = {name: make_model(name) for name in MODEL_CLASSES}

In [3]:
def metrics(y, p):
    return {"rmse": np.sqrt(mean_squared_error(y, p)),
            "mae": mean_absolute_error(y, p),
            "r2": r2_score(y, p)}

oof = {name: cross_val_predict(m, X_train, y_train, cv=cv) for name, m in models.items()}

oof_scores = pd.DataFrame({name: metrics(y_train, p) for name, p in oof.items()}).T
oof_scores.round(3)

,rmse,mae,r2
Random Forest,0.740,0.540,0.772
Extra Trees,0.743,0.529,0.770
Gradient Boosting,0.735,0.525,0.775
XGBoost,0.733,0.542,0.776
LightGBM,0.731,0.521,0.778
CatBoost,0.696,0.490,0.799


In [4]:
M = oof_scores[["rmse", "mae", "r2"]].values
benefit = np.array([False, False, True])   # rmse, mae: lower is better; r2: higher is better

# Step A: entropy -> how much each metric matters
P = M / M.sum(axis=0)
E = -(P * np.log(P)).sum(axis=0) / np.log(len(M))
metric_w = (1 - E) / (1 - E).sum()

# Step B: TOPSIS -> how close each model is to the "perfect model"
R = M / np.sqrt((M ** 2).sum(axis=0))
V = R * metric_w
ideal = np.where(benefit, V.max(axis=0), V.min(axis=0))
anti  = np.where(benefit, V.min(axis=0), V.max(axis=0))
d_ideal = np.sqrt(((V - ideal) ** 2).sum(axis=1))
d_anti  = np.sqrt(((V - anti) ** 2).sum(axis=1))
closeness = d_anti / (d_ideal + d_anti)

weights = pd.Series(closeness / closeness.sum(), index=oof_scores.index)
print("Metric weights:", dict(zip(["rmse", "mae", "r2"], metric_w.round(3))))
weights.sort_values(ascending=False).round(3)

Metric weights: {'rmse': np.float64(0.281), 'mae': np.float64(0.633), 'r2': np.float64(0.086)}


CatBoost             0.492
LightGBM             0.194
Gradient Boosting    0.154
Extra Trees          0.114
XGBoost              0.028
Random Forest        0.019
dtype: float64

In [5]:
val_preds = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    val_preds[name] = m.predict(X_val)

ens_weighted = sum(weights[n] * val_preds[n] for n in weights.index)
ens_average = np.mean(list(val_preds.values()), axis=0)

rows = {name: metrics(y_val, p) for name, p in val_preds.items()}
rows["Ensemble (entropy-TOPSIS)"] = metrics(y_val, ens_weighted)
rows["Ensemble (simple average)"] = metrics(y_val, ens_average)

val_table = pd.DataFrame(rows).T.sort_values("rmse").round(3)
val_table

,rmse,mae,r2
Ensemble (entropy-TOPSIS),0.612,0.373,0.835
XGBoost,0.612,0.387,0.835
CatBoost,0.617,0.389,0.832
Ensemble (simple average),0.618,0.379,0.832
Extra Trees,0.640,0.416,0.820
Gradient Boosting,0.641,0.402,0.819
Random Forest,0.656,0.439,0.810
LightGBM,0.671,0.414,0.801


In [6]:
with open(ARTIFACTS / "ensemble_weights.json", "w") as f:
    json.dump(weights.round(4).to_dict(), f, indent=2)

oof_scores.round(3).to_csv(RESULTS / "06_oof_scores.csv")
val_table.to_csv(RESULTS / "06_validation_comparison.csv")
print("saved")

saved
